# 10 Guardar imagenes a escala

Este notebook ejecuta `preprocesar_he_hsi()` para todos los sujetos (`SB012`, `SB013`, `SB017`, `SB018`, `SB019`, `SB020`) y guarda los artefactos necesarios para preparar el registro H&E -> HSI:

- RGB H&E y HSI segmentadas a escala real.
- Mascaras binarias escaladas en el mismo sistema de pixeles que cada RGB.
- Contornos de las mascaras para inicializacion por forma/rotacion.
- Mapas de distancia firmada (`.npy`) para registro basado en forma.
- Metadatos por sujeto y un `manifest.json` global con rutas, escala y bboxes.

Las salidas se guardan en:

`Registration\DeepLearning\Imagenes_a_escala`

Si se quiere meter un nuevo sujeto

Nuevo sujeto:
1. añadir rutas H&E/HSI a SPECIMENS
2. añadir ID a SUBJECTS
3. si necesita grayscale:
      añadir ID a GRAYSCALE_HSI_SUBJECTS
      añadir ROI a WIDE_MASK_TRAY_CONFIGS
   si no:
      dejar que use caja azul
4. ejecutar 10_guardar_imagenes_a_escala.ipynb

In [ ]:
from pathlib import Path
from PIL import Image
import json
import nbformat
import numpy as np

BASE_DIR = Path(r'Registration\DeepLearning')
PREPROCESS_NOTEBOOK = BASE_DIR / '9_funcion_preprocesado.ipynb'
OUTPUT_DIR = BASE_DIR / 'Imagenes_a_escala'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Notebook de funciones:', PREPROCESS_NOTEBOOK)
print('Carpeta de salida:', OUTPUT_DIR)


## Cargar funciones del notebook 9

Se cargan solo las celdas de definiciones y se saltan las celdas de ejemplo/validacion para no generar plots ni salidas extra.

In [ ]:
def load_preprocess_definitions(notebook_path):
    nb = nbformat.read(str(notebook_path), as_version=4)
    skipped = []

    for idx, cell in enumerate(nb.cells):
        if cell.cell_type != 'code':
            continue

        source = cell.source
        skip_patterns = [
            'preprocess_output = preprocesar_he_hsi',
            'validation_results = {}',
        ]
        if any(pattern in source for pattern in skip_patterns):
            skipped.append(idx)
            continue

        exec(compile(source, f'{notebook_path.name}:cell_{idx}', 'exec'), globals())

    print(f'Celdas de ejemplo/validacion saltadas: {skipped}')


load_preprocess_definitions(PREPROCESS_NOTEBOOK)


## Segmentacion validada para el guardado final

Estas funciones sustituyen localmente a las del notebook 9 para que `preprocesar_he_hsi()` use las mascaras revisadas en `Comprobacion_H&E.ipynb` y `Comprobacion_HSI.ipynb` sin modificar el notebook base.


In [ ]:
# Overrides validados en Comprobacion_H&E y Comprobacion_HSI
import warnings


def _largest_component_he_scale(mask, min_area=500):
    mask_u8 = (np.asarray(mask) > 0).astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if num_labels <= 1:
        return np.zeros(mask_u8.shape, dtype=bool)

    areas = stats[1:, cv2.CC_STAT_AREA]
    valid = np.where(areas >= min_area)[0]
    if len(valid) > 0:
        largest_label = 1 + valid[np.argmax(areas[valid])]
    else:
        largest_label = 1 + int(np.argmax(areas))
    return labels == largest_label


def extract_he_tissue_contour_image(
    slide_path,
    max_dim=1800,
    sat_thresh=10,
    val_thresh=252,
    od_thresh=0.025,
    chroma_thresh=4.0,
    min_area=500,
    open_kernel_size=3,
    close_kernel_size=65,
    grow_pixels=5,
    use_convex_hull=False,
    show_original=True,
    show_result=True,
    debug=False,
    return_mask=False,
    return_info=False,
):
    """Segmenta H&E con semillas de tincion rosa/morada, evitando fondo blanco poco saturado."""
    slide_path = Path(slide_path)
    if not slide_path.exists():
        raise FileNotFoundError(f'No existe la H&E: {slide_path}')

    rgb_u8 = np.asarray(Image.open(slide_path).convert('RGB'), dtype=np.uint8)
    level_info = read_he_slidedat_info(slide_path)
    chosen_level = level_info['preview_level']
    level_w, level_h = rgb_u8.shape[1], rgb_u8.shape[0]

    hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]

    lab = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2LAB).astype(np.float32)
    lab_l = lab[:, :, 0]
    lab_a = lab[:, :, 1] - 128.0
    lab_b = lab[:, :, 2] - 128.0
    chroma = np.sqrt(lab_a * lab_a + lab_b * lab_b)

    mask_color = ((sat > sat_thresh) & (val < val_thresh)) | ((chroma > chroma_thresh) & (lab_l < 252))
    mask = mask_color.astype(np.uint8) * 255

    if open_kernel_size and open_kernel_size > 1:
        kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_kernel_size, open_kernel_size))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)

    if close_kernel_size and close_kernel_size > 1:
        kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel_size, close_kernel_size))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    mask_clean = _largest_component_he_scale(mask, min_area=min_area).astype(np.uint8) * 255

    contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask_final = np.zeros_like(mask_clean, dtype=np.uint8)
    if contours:
        tissue_contour = max(contours, key=cv2.contourArea)
        if use_convex_hull:
            tissue_contour = cv2.convexHull(tissue_contour)
        cv2.drawContours(mask_final, [tissue_contour], -1, 255, thickness=cv2.FILLED)

    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * grow_pixels + 1, 2 * grow_pixels + 1))
        mask_final = cv2.dilate(mask_final, kernel_grow, iterations=1)

    tissue_only_rgb = rgb_u8.copy()
    tissue_only_rgb[mask_final == 0] = 0

    if debug:
        fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
        panels = [
            (rgb_u8, f'H&E original | PIL preview level={chosen_level}', None),
            (sat, 'Saturacion HSV', 'gray'),
            (chroma, 'Croma LAB', 'gray'),
            (mask, 'Mascara inicial tincion', 'gray'),
            (mask_final, 'Mascara final por contorno', 'gray'),
            (tissue_only_rgb, 'Solo tejido', None),
        ]
        for ax, (img, title, cmap) in zip(axes.ravel(), panels):
            ax.imshow(img, cmap=cmap)
            ax.set_title(title)
            ax.axis('off')
        plt.show()
    elif show_original and show_result:
        fig, axes = plt.subplots(1, 2, figsize=(10, 6), constrained_layout=True)
        axes[0].imshow(rgb_u8)
        axes[0].set_title(f'H&E original | PIL preview level={chosen_level}')
        axes[0].axis('off')
        axes[1].imshow(tissue_only_rgb)
        axes[1].set_title('Solo tejido')
        axes[1].axis('off')
        plt.show()
    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f'H&E original | PIL preview level={chosen_level}')
        plt.axis('off')
        plt.show()
    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(tissue_only_rgb)
        plt.title('Solo tejido')
        plt.axis('off')
        plt.show()

    outputs = [tissue_only_rgb]
    if return_mask:
        outputs.append(mask_final)
    if return_info:
        outputs.append({
            'method': 'he_stain_chroma_mask',
            'chosen_level': int(chosen_level),
            'read_dimensions': (int(level_w), int(level_h)),
            'sat_thresh': sat_thresh,
            'val_thresh': val_thresh,
            'od_thresh_ignored': od_thresh,
            'chroma_thresh': chroma_thresh,
            'close_kernel_size': close_kernel_size,
            'grow_pixels': grow_pixels,
            'use_convex_hull': bool(use_convex_hull),
            'mask_area_px': int(np.count_nonzero(mask_final)),
            'reader': 'PIL',
            'slidedat_path': level_info['slidedat_path'],
        })
    return outputs[0] if len(outputs) == 1 else tuple(outputs)


def _hsi_subject_id_from_path(hdr_path):
    path_upper = str(hdr_path).upper()
    for subject_id in ['SB012', 'SB013', 'SB017', 'SB018', 'SB019', 'SB020']:
        if subject_id in path_upper:
            return subject_id
    return None


def _robust_normalize_for_display(channel, p_low=2, p_high=98, gamma=1.0):
    ch = np.asarray(channel, dtype=np.float32)
    ch = np.where(np.isfinite(ch), ch, np.nan)
    if np.all(np.isnan(ch)):
        return np.zeros_like(ch, dtype=np.float32)
    lo = np.nanpercentile(ch, p_low)
    hi = np.nanpercentile(ch, p_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(ch, dtype=np.float32)
    ch = np.clip((ch - lo) / (hi - lo), 0, 1)
    if gamma != 1.0:
        ch = ch ** gamma
    return np.nan_to_num(ch, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)


def _load_hsi_cube_and_wavelengths(hdr_path):
    img = open_image(str(hdr_path))
    try:
        from spectral.io.spyfile import NaNValueWarning
        ignored_warnings = (NaNValueWarning,)
    except Exception:
        ignored_warnings = (Warning,)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ignored_warnings)
        cube = np.asarray(img.load(), dtype=np.float32)

    wavelengths = np.array([float(w) for w in img.metadata['wavelength']], dtype=np.float32)
    return cube, wavelengths


def _load_hsi_grayscale_for_registration(hdr_path, min_nm=435, max_nm=1700, p_low=0.1, p_high=99.9, gamma=0.70):
    cube, wavelengths = _load_hsi_cube_and_wavelengths(hdr_path)
    band_idx = np.where((wavelengths >= min_nm) & (wavelengths <= max_nm))[0]
    if band_idx.size == 0:
        band_idx = np.arange(cube.shape[2])

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        gray_float = np.nanmean(cube[:, :, band_idx], axis=2)

    gray_float = np.nan_to_num(gray_float, nan=0.0, posinf=0.0, neginf=0.0)
    gray_u8 = (_robust_normalize_for_display(
        gray_float,
        p_low=p_low,
        p_high=p_high,
        gamma=gamma,
    ) * 255).astype(np.uint8)

    info = {
        'display_mode': 'hyperlab_full_range' if min_nm <= 436 and max_nm >= 1700 else 'wide_grayscale',
        'shape': gray_u8.shape,
        'min_nm': float(wavelengths[band_idx[0]]),
        'max_nm': float(wavelengths[band_idx[-1]]),
        'num_bands': int(band_idx.size),
        'p_low': float(p_low),
        'p_high': float(p_high),
        'gamma': float(gamma),
    }
    return gray_u8, info


def _gray_to_rgb_registration(gray_u8):
    return np.repeat(np.asarray(gray_u8, dtype=np.uint8)[:, :, None], 3, axis=2)


def _largest_component_hsi_registration(mask):
    mask_u8 = (np.asarray(mask) > 0).astype(np.uint8)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if num_labels <= 1:
        return mask_u8.astype(bool)

    h, w = mask_u8.shape
    center = np.array([w / 2, h / 2], dtype=np.float32)
    best_label = 1
    best_score = -np.inf
    for label in range(1, num_labels):
        area = stats[label, cv2.CC_STAT_AREA]
        if area < 1500:
            continue
        x = stats[label, cv2.CC_STAT_LEFT]
        y = stats[label, cv2.CC_STAT_TOP]
        bw = stats[label, cv2.CC_STAT_WIDTH]
        bh = stats[label, cv2.CC_STAT_HEIGHT]
        distance = np.linalg.norm(centroids[label] - center) / max(h, w)
        is_strip = (bw > 0.92 * w and bh < 0.25 * h) or (bh > 0.92 * h and bw < 0.25 * w)
        score = area * (1 - 0.2 * distance) * (0.3 if is_strip else 1.0)
        if score > best_score:
            best_score = score
            best_label = label

    return labels == best_label


GRAYSCALE_HSI_SUBJECTS = {'SB017', 'SB018', 'SB019', 'SB020'}
REFERENCE_HSI_SIZE = (1024, 1280)
WIDE_MASK_TRAY_CONFIGS = {
    'SB017': {'roi': (460, 260, 1000, 790), 'border': 10, 'threshold_factor': 0.55},
    'SB018': {'roi': (460, 265, 1000, 790), 'border': 10, 'threshold_factor': 0.55},
    'SB019': {'roi': (450, 275, 1010, 790), 'border': 10, 'threshold_factor': 0.55},
    'SB020': {'roi': (455, 280, 1000, 790), 'border': 10, 'threshold_factor': 0.55},
}


def _scale_roi_to_image(roi, image_shape):
    ref_h, ref_w = REFERENCE_HSI_SIZE
    h, w = image_shape[:2]
    x1, y1, x2, y2 = roi
    scaled = (
        int(round(x1 * w / ref_w)),
        int(round(y1 * h / ref_h)),
        int(round(x2 * w / ref_w)),
        int(round(y2 * h / ref_h)),
    )
    sx1, sy1, sx2, sy2 = scaled
    sx1 = max(0, min(w - 2, sx1))
    sx2 = max(sx1 + 2, min(w, sx2))
    sy1 = max(0, min(h - 2, sy1))
    sy2 = max(sy1 + 2, min(h, sy2))
    return sx1, sy1, sx2, sy2


def _segment_hsi_grayscale_tray_for_registration(gray_u8, subject_id):
    if subject_id not in WIDE_MASK_TRAY_CONFIGS:
        raise ValueError(f'No hay configuracion grayscale definida para {subject_id}')

    config = WIDE_MASK_TRAY_CONFIGS[subject_id]
    x1, y1, x2, y2 = _scale_roi_to_image(config['roi'], gray_u8.shape)
    roi = gray_u8[y1:y2, x1:x2]
    valid_values = roi[(roi > 5) & (roi < 250)]
    if valid_values.size > 100:
        otsu_value, _ = cv2.threshold(
            valid_values.reshape(-1, 1).astype(np.uint8),
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU,
        )
    else:
        otsu_value = 80.0

    threshold_factor = float(config.get('threshold_factor', 0.70))
    threshold = max(34.0, min(95.0, float(otsu_value) * threshold_factor))
    roi_mask = (roi > threshold) & (roi < 245)

    border = max(6, int(round(config.get('border', 10) * min(roi.shape[:2]) / 520)))
    roi_mask[:border, :] = False
    roi_mask[-border:, :] = False
    roi_mask[:, :border] = False
    roi_mask[:, -border:] = False

    open_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    close_kernel_1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17))
    close_kernel_2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
    dilate_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    roi_mask = cv2.morphologyEx(roi_mask.astype(np.uint8) * 255, cv2.MORPH_OPEN, open_kernel) > 0
    roi_mask = cv2.morphologyEx(roi_mask.astype(np.uint8) * 255, cv2.MORPH_CLOSE, close_kernel_1) > 0
    roi_mask = _largest_component_hsi_registration(roi_mask)
    roi_mask = cv2.morphologyEx(roi_mask.astype(np.uint8) * 255, cv2.MORPH_CLOSE, close_kernel_2) > 0
    roi_mask = cv2.dilate(roi_mask.astype(np.uint8), dilate_kernel, iterations=1) > 0

    final_border = max(2, border // 6)
    roi_mask[:final_border, :] = False
    roi_mask[-final_border:, :] = False
    roi_mask[:, :final_border] = False
    roi_mask[:, -final_border:] = False

    mask = np.zeros(gray_u8.shape, dtype=bool)
    mask[y1:y2, x1:x2] = roi_mask

    info = {
        'selected_method': 'hyperlab_gray_roi_mask',
        'selected_because': 'same_hyperlab_gray_display_used_for_contour',
        'roi_xyxy': (int(x1), int(y1), int(x2), int(y2)),
        'otsu_value': float(otsu_value),
        'threshold': float(threshold),
        'threshold_factor': threshold_factor,
        'mask_area_px': int(np.count_nonzero(mask)),
    }
    return mask, info


if '_extract_specimen_only_from_hsi_path_original' not in globals():
    _extract_specimen_only_from_hsi_path_original = extract_specimen_only_from_hsi_path


def extract_specimen_only_from_hsi_path(
    hdr_path,
    r_nm=650,
    g_nm=550,
    b_nm=450,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=False,
    return_info=False,
):
    """Usa segmentacion grayscale HyperLAB-like para SB017-SB020 y conserva el metodo original para SB012-SB013."""
    subject_id = _hsi_subject_id_from_path(hdr_path)

    if subject_id not in GRAYSCALE_HSI_SUBJECTS:
        return _extract_specimen_only_from_hsi_path_original(
            hdr_path,
            r_nm=r_nm,
            g_nm=g_nm,
            b_nm=b_nm,
            grow_pixels=grow_pixels,
            show_original=show_original,
            show_result=show_result,
            return_mask=return_mask,
            return_info=return_info,
        )

    gray_u8, gray_info = _load_hsi_grayscale_for_registration(
        hdr_path,
        min_nm=435,
        max_nm=1700,
        p_low=0.1,
        p_high=99.9,
        gamma=0.70,
    )
    original_rgb = _gray_to_rgb_registration(gray_u8)
    mask, seg_info = _segment_hsi_grayscale_tray_for_registration(gray_u8, subject_id)
    specimen_rgb = original_rgb.copy()
    specimen_rgb[~mask] = 0

    seg_info = dict(seg_info)
    seg_info.update({
        'display_mode': gray_info['display_mode'],
        'grayscale_info': gray_info,
        'mask_grayscale_info': gray_info,
        'subject_id': subject_id,
        'grow_pixels_requested': int(grow_pixels),
        'final_dilation_kernel': (5, 5),
    })

    if show_original and show_result:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
        axes[0].imshow(original_rgb)
        axes[0].set_title(f'{subject_id} HSI - HyperLAB-like grayscale original')
        axes[0].axis('off')
        axes[1].imshow(specimen_rgb)
        axes[1].set_title(f'{subject_id} HSI - segmentada')
        axes[1].axis('off')
        plt.show()
    elif show_original:
        plt.figure(figsize=(6, 5))
        plt.imshow(original_rgb)
        plt.title(f'{subject_id} HSI - HyperLAB-like grayscale original')
        plt.axis('off')
        plt.show()
    elif show_result:
        plt.figure(figsize=(6, 5))
        plt.imshow(specimen_rgb)
        plt.title(f'{subject_id} HSI - segmentada')
        plt.axis('off')
        plt.show()

    outputs = [specimen_rgb]
    if return_mask:
        outputs.append(mask)
    if return_info:
        outputs.append(seg_info)
    return outputs[0] if len(outputs) == 1 else tuple(outputs)


print('Overrides de segmentacion activos: H&E=he_stain_chroma_mask; HSI SB017-SB020=hyperlab_gray_roi_mask')


## Guardar artefactos para registration


In [ ]:
SUBJECTS = ['SB012', 'SB013', 'SB017', 'SB018', 'SB019', 'SB020']


def to_uint8_rgb(img):
    arr = np.asarray(img)
    if arr.dtype != np.uint8:
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    if arr.ndim == 2:
        arr = np.repeat(arr[:, :, None], 3, axis=2)
    return arr


def crop_mask_xyxy(mask, bbox):
    x1, y1, x2, y2 = [int(v) for v in bbox]
    return (np.asarray(mask) > 0)[y1:y2 + 1, x1:x2 + 1]


def resize_mask_to_rgb(mask, rgb):
    target_h, target_w = np.asarray(rgb).shape[:2]
    mask_u8 = (np.asarray(mask) > 0).astype(np.uint8) * 255
    resized = cv2.resize(mask_u8, (target_w, target_h), interpolation=cv2.INTER_NEAREST)
    return resized > 0


def mask_to_uint8(mask):
    return (np.asarray(mask) > 0).astype(np.uint8) * 255


def mask_contour(mask, thickness=2):
    mask_u8 = mask_to_uint8(mask)
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    out = np.zeros_like(mask_u8)
    cv2.drawContours(out, contours, contourIdx=-1, color=255, thickness=int(thickness))
    return out


def signed_distance_from_mask(mask):
    mask_u8 = (np.asarray(mask) > 0).astype(np.uint8)
    inside = cv2.distanceTransform(mask_u8, cv2.DIST_L2, 5)
    outside = cv2.distanceTransform(1 - mask_u8, cv2.DIST_L2, 5)
    return (inside - outside).astype(np.float32)


def save_rgb(path, img):
    Image.fromarray(to_uint8_rgb(img)).save(path)
    return path


def save_mask(path, mask):
    Image.fromarray(mask_to_uint8(mask)).save(path)
    return path


def json_ready(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    return value


def build_pair_preview(he_img, hsi_img, he_mask, hsi_mask):
    he = to_uint8_rgb(he_img).copy()
    hsi = to_uint8_rgb(hsi_img).copy()
    he_contour = mask_contour(he_mask, thickness=2) > 0
    hsi_contour = mask_contour(hsi_mask, thickness=2) > 0
    he[he_contour] = np.array([0, 255, 255], dtype=np.uint8)
    hsi[hsi_contour] = np.array([0, 255, 255], dtype=np.uint8)

    target_h = max(he.shape[0], hsi.shape[0])

    def pad_to_height(img, height):
        pad_h = height - img.shape[0]
        if pad_h <= 0:
            return img
        top = pad_h // 2
        bottom = pad_h - top
        return np.pad(img, ((top, bottom), (0, 0), (0, 0)), mode='constant', constant_values=0)

    gap = np.zeros((target_h, 20, 3), dtype=np.uint8)
    return np.concatenate([pad_to_height(he, target_h), gap, pad_to_height(hsi, target_h)], axis=1)


saved_outputs = {}
manifest = {
    'registration_direction': 'he_to_hsi',
    'fixed_image': 'hsi',
    'moving_image': 'he',
    'notes': [
        'Las RGB ya estan segmentadas y reescaladas a una escala fisica comun.',
        'Las mascaras, contornos y mapas de distancia tienen las mismas dimensiones que su RGB correspondiente.',
        'Los mapas signed_distance estan en pixeles: positivo dentro del especimen y negativo fuera.',
    ],
    'subjects': {},
}

for subject_id in SUBJECTS:
    print(f'=== {subject_id} ===')
    paths = SPECIMENS[subject_id]
    subject_dir = OUTPUT_DIR / subject_id
    subject_dir.mkdir(parents=True, exist_ok=True)

    output = preprocesar_he_hsi(
        he_path=paths['he_path'],
        hsi_hdr_path=paths['hsi_hdr_path'],
        specimen_id=subject_id,
        show_he_basica=False,
        show_print_he_datos=False,
        show_hsi_basica=False,
        show_hsi_contorno_caja=False,
        show_hsi_contorno_medidas=False,
        show_lienzo_unico_caja=False,
        show_he_segmentada=False,
        show_hsi_segmentada=False,
        show_lienzo_unico_segmentadas=False,
        show_he_segmentada_escala=False,
        show_hsi_segmentada_escala=False,
        show_he_segmentada_escala_sin_barra=False,
        show_hsi_segmentada_escala_sin_barra=False,
    )

    he_img = output['he_segmentada_escala_rgb']
    hsi_img = output['hsi_segmentada_escala_rgb']
    segmented_info = output['same_scale_segmented_output']['info']

    he_mask_crop = crop_mask_xyxy(output['he_tissue_mask'], segmented_info['he_bbox_xyxy'])
    hsi_mask_crop = crop_mask_xyxy(output['hsi_specimen_mask'], segmented_info['hsi_specimen_bbox_xyxy'])
    he_mask = resize_mask_to_rgb(he_mask_crop, he_img)
    hsi_mask = resize_mask_to_rgb(hsi_mask_crop, hsi_img)

    he_contour = mask_contour(he_mask)
    hsi_contour = mask_contour(hsi_mask)
    he_signed_distance = signed_distance_from_mask(he_mask)
    hsi_signed_distance = signed_distance_from_mask(hsi_mask)
    preview = build_pair_preview(he_img, hsi_img, he_mask, hsi_mask)

    files = {
        'he_rgb': subject_dir / f'{subject_id}_h&e.png',
        'hsi_rgb': subject_dir / f'{subject_id}_hsi.png',
        'he_mask': subject_dir / f'{subject_id}_he_mask.png',
        'hsi_mask': subject_dir / f'{subject_id}_hsi_mask.png',
        'he_contour': subject_dir / f'{subject_id}_he_contour.png',
        'hsi_contour': subject_dir / f'{subject_id}_hsi_contour.png',
        'he_signed_distance': subject_dir / f'{subject_id}_he_signed_distance.npy',
        'hsi_signed_distance': subject_dir / f'{subject_id}_hsi_signed_distance.npy',
        'pair_preview': subject_dir / f'{subject_id}_pair_preview.png',
        'metadata': subject_dir / f'{subject_id}_metadata.json',
    }

    save_rgb(files['he_rgb'], he_img)
    save_rgb(files['hsi_rgb'], hsi_img)
    save_mask(files['he_mask'], he_mask)
    save_mask(files['hsi_mask'], hsi_mask)
    Image.fromarray(he_contour).save(files['he_contour'])
    Image.fromarray(hsi_contour).save(files['hsi_contour'])
    np.save(files['he_signed_distance'], he_signed_distance)
    np.save(files['hsi_signed_distance'], hsi_signed_distance)
    save_rgb(files['pair_preview'], preview)

    subject_metadata = {
        'subject_id': subject_id,
        'registration_direction': 'he_to_hsi',
        'fixed_image': 'hsi',
        'moving_image': 'he',
        'source_paths': {
            'he_path': paths['he_path'],
            'hsi_hdr_path': paths['hsi_hdr_path'],
        },
        'files': files,
        'shapes': {
            'he_rgb': list(np.asarray(he_img).shape),
            'hsi_rgb': list(np.asarray(hsi_img).shape),
            'he_mask': list(he_mask.shape),
            'hsi_mask': list(hsi_mask.shape),
        },
        'mask_area_px': {
            'he': int(np.count_nonzero(he_mask)),
            'hsi': int(np.count_nonzero(hsi_mask)),
        },
        'target_px_per_cm': segmented_info['target_px_per_cm'],
        'segmented_info': segmented_info,
        'he_scale_info': output['he_scale_info'],
        'he_segmentation_info': output.get('he_tissue_info'),
        'hsi_measure_info': output['measure_info'],
        'hsi_segmentation_info': output['same_scale_segmented_output'].get('hsi_segmentation_info'),
    }

    files['metadata'].write_text(
        json.dumps(json_ready(subject_metadata), indent=2, ensure_ascii=False),
        encoding='utf-8',
    )

    saved_outputs[subject_id] = {
        'he_path': files['he_rgb'],
        'hsi_path': files['hsi_rgb'],
        'he_mask_path': files['he_mask'],
        'hsi_mask_path': files['hsi_mask'],
        'metadata_path': files['metadata'],
        'he_shape': np.asarray(he_img).shape,
        'hsi_shape': np.asarray(hsi_img).shape,
        'target_px_per_cm': segmented_info['target_px_per_cm'],
    }
    manifest['subjects'][subject_id] = subject_metadata

    print('Guardado H&E:', files['he_rgb'])
    print('Guardado HSI:', files['hsi_rgb'])
    print('Guardadas mascaras:', files['he_mask'], '|', files['hsi_mask'])
    print('Guardados metadatos:', files['metadata'])

manifest_path = OUTPUT_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(json_ready(manifest), indent=2, ensure_ascii=False), encoding='utf-8')
print('Manifest global:', manifest_path)

saved_outputs


In [ ]:
# Figura para la memoria: segmentación HSI de SB017 mediante una ROI definida en Python
figure_subject = 'SB017'
figure_hsi_path = SPECIMENS[figure_subject]['hsi_hdr_path']

gray_u8, gray_info = _load_hsi_grayscale_for_registration(
    figure_hsi_path,
    min_nm=435,
    max_nm=1700,
    p_low=0.1,
    p_high=99.9,
    gamma=0.70,
)
mask, segmentation_info = _segment_hsi_grayscale_tray_for_registration(
    gray_u8,
    figure_subject,
)

original_rgb = _gray_to_rgb_registration(gray_u8)
roi_overlay = original_rgb.copy()
x1, y1, x2, y2 = segmentation_info['roi_xyxy']
cv2.rectangle(roi_overlay, (x1, y1), (x2 - 1, y2 - 1), (255, 255, 0), 4)

segmented_rgb = original_rgb.copy()
segmented_rgb[~mask] = 0
contour_overlay = segmented_rgb.copy()
contours, _ = cv2.findContours(
    mask.astype(np.uint8) * 255,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE,
)
if contours:
    cv2.drawContours(
        contour_overlay,
        [max(contours, key=cv2.contourArea)],
        -1,
        (0, 255, 255),
        3,
    )

fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
axes[0].imshow(original_rgb)
axes[0].set_title('Representación en escala de grises')
axes[1].imshow(roi_overlay)
axes[1].set_title('ROI definida en Python')
axes[2].imshow(mask, cmap='gray')
axes[2].set_title('Máscara refinada')
axes[3].imshow(contour_overlay)
axes[3].set_title('Espécimen segmentado y contorno')
for ax in axes:
    ax.axis('off')

figure_path = BASE_DIR.parents[1] / 'Fotos_Memoria' / 'SB017_segmentacion_HSI_ROI.png'
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Figura guardada en:', figure_path)


In [ ]:
# Figura para la memoria: mascaras binarias y mapas de distancia firmados de SB013
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageOps


figure_subject = 'SB013'
notebook_dir = Path.cwd()
if not (notebook_dir / 'Imagenes_a_escala').exists():
    notebook_dir = notebook_dir / 'Registration' / 'DeepLearning'

subject_dir = notebook_dir / 'Imagenes_a_escala' / figure_subject
figure_path = notebook_dir.parents[1] / 'Fotos_Memoria' / 'SB013_mascaras_mapas_distancia.png'

he_mask = np.asarray(Image.open(subject_dir / 'SB013_he_mask.png').convert('L')) > 0
hsi_mask = np.asarray(Image.open(subject_dir / 'SB013_hsi_mask.png').convert('L')) > 0
he_sdf = np.load(subject_dir / 'SB013_he_signed_distance.npy').astype(np.float32)
hsi_sdf = np.load(subject_dir / 'SB013_hsi_signed_distance.npy').astype(np.float32)

# Escala visual comun y robusta. Los .npy conservan las distancias originales en pixeles.
distance_limit = float(np.percentile(np.concatenate([
    np.abs(he_sdf).ravel(),
    np.abs(hsi_sdf).ravel(),
]), 98))
distance_limit = max(distance_limit, 1.0)


def diverging_rgb(values, limit):
    normalized = np.clip(np.asarray(values, dtype=np.float32) / limit, -1.0, 1.0)
    output = np.full((*normalized.shape, 3), 255.0, dtype=np.float32)
    negative_color = np.array([49.0, 54.0, 149.0], dtype=np.float32)
    positive_color = np.array([165.0, 0.0, 38.0], dtype=np.float32)

    negative = normalized < 0
    positive = normalized > 0
    if np.any(negative):
        weight = (-normalized[negative])[:, None]
        output[negative] = 255.0 * (1.0 - weight) + negative_color * weight
    if np.any(positive):
        weight = normalized[positive][:, None]
        output[positive] = 255.0 * (1.0 - weight) + positive_color * weight
    return np.clip(output, 0, 255).astype(np.uint8)


def load_font(size, bold=False):
    candidates = [
        Path(r'C:\Windows\Fonts\arialbd.ttf' if bold else r'C:\Windows\Fonts\arial.ttf'),
        Path(r'C:\Windows\Fonts\calibrib.ttf' if bold else r'C:\Windows\Fonts\calibri.ttf'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return ImageFont.truetype(str(candidate), size=size)
    return ImageFont.load_default()


def centered_text(draw, box, text, font, fill=(25, 25, 25)):
    x1, y1, x2, y2 = box
    bounds = draw.textbbox((0, 0), text, font=font)
    width = bounds[2] - bounds[0]
    height = bounds[3] - bounds[1]
    draw.text(((x1 + x2 - width) / 2, (y1 + y2 - height) / 2 - bounds[1]), text, font=font, fill=fill)


def paste_panel(canvas, source, box, title, title_font):
    x1, y1, x2, y2 = box
    title_height = 54
    image_box = (x1 + 8, y1 + title_height, x2 - 8, y2 - 8)
    image_width = image_box[2] - image_box[0]
    image_height = image_box[3] - image_box[1]
    panel_image = Image.fromarray(source).convert('RGB')
    panel_image = ImageOps.contain(panel_image, (image_width, image_height), Image.Resampling.LANCZOS)
    px = image_box[0] + (image_width - panel_image.width) // 2
    py = image_box[1] + (image_height - panel_image.height) // 2
    canvas.paste(panel_image, (px, py))
    draw = ImageDraw.Draw(canvas)
    draw.rectangle(box, outline=(205, 205, 205), width=2)
    centered_text(draw, (x1, y1, x2, y1 + title_height), title, title_font)


mask_he_rgb = np.repeat((he_mask.astype(np.uint8) * 255)[:, :, None], 3, axis=2)
mask_hsi_rgb = np.repeat((hsi_mask.astype(np.uint8) * 255)[:, :, None], 3, axis=2)
sdf_he_rgb = diverging_rgb(he_sdf, distance_limit)
sdf_hsi_rgb = diverging_rgb(hsi_sdf, distance_limit)

canvas = Image.new('RGB', (1800, 1500), 'white')
title_font = load_font(34, bold=False)
label_font = load_font(28, bold=False)
small_font = load_font(24, bold=False)

panel_boxes = [
    (55, 35, 865, 650),
    (935, 35, 1745, 650),
    (55, 700, 865, 1315),
    (935, 700, 1745, 1315),
]
panel_data = [
    (mask_he_rgb, '(a) M\u00e1scara H&E m\u00f3vil'),
    (mask_hsi_rgb, '(b) M\u00e1scara HSI fija'),
    (sdf_he_rgb, '(c) Mapa de distancia H&E'),
    (sdf_hsi_rgb, '(d) Mapa de distancia HSI'),
]
for box, (panel, title) in zip(panel_boxes, panel_data):
    paste_panel(canvas, panel, box, title, title_font)

draw = ImageDraw.Draw(canvas)
bar_x1, bar_y1, bar_x2, bar_y2 = 400, 1360, 1400, 1400
gradient_values = np.linspace(-distance_limit, distance_limit, bar_x2 - bar_x1, dtype=np.float32)[None, :]
gradient_rgb = diverging_rgb(gradient_values, distance_limit)
gradient_image = Image.fromarray(gradient_rgb).resize((bar_x2 - bar_x1, bar_y2 - bar_y1))
canvas.paste(gradient_image, (bar_x1, bar_y1))
draw.rectangle((bar_x1, bar_y1, bar_x2, bar_y2), outline=(80, 80, 80), width=2)

limit_label = f'{distance_limit:.1f} px'
centered_text(draw, (bar_x1 - 100, bar_y1, bar_x1 + 100, bar_y2 + 60), f'-{limit_label}', small_font)
centered_text(draw, (bar_x2 - 100, bar_y1, bar_x2 + 100, bar_y2 + 60), f'+{limit_label}', small_font)
centered_text(draw, (850, bar_y1, 950, bar_y2 + 60), '0', small_font)
centered_text(
    draw,
    (300, 1420, 1500, 1485),
    'Distancia firmada (p\u00edxeles): exterior < 0, interior > 0',
    label_font,
)

figure_path.parent.mkdir(parents=True, exist_ok=True)
canvas.save(figure_path, dpi=(300, 300))

print('Figura guardada en:', figure_path)
print('L\u00edmite sim\u00e9trico de visualizaci\u00f3n (percentil 98):', round(distance_limit, 2), 'px')


In [ ]:
# Figura 19 para la memoria: salidas finales de SB013 a escala fisica comun
import json
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont


figure_subject = 'SB013'
notebook_dir = Path.cwd()
if not (notebook_dir / 'Imagenes_a_escala').exists():
    notebook_dir = notebook_dir / 'Registration' / 'DeepLearning'

subject_dir = notebook_dir / 'Imagenes_a_escala' / figure_subject
figure_path = notebook_dir.parents[1] / 'Fotos_Memoria' / 'Escala_1cm_SB013.png'

he_image = Image.open(subject_dir / 'SB013_h&e.png').convert('RGB')
hsi_image = Image.open(subject_dir / 'SB013_hsi.png').convert('RGB')
metadata = json.loads((subject_dir / 'SB013_metadata.json').read_text(encoding='utf-8'))
px_per_cm = float(metadata['target_px_per_cm'])

expected_he_size = tuple(reversed(metadata['shapes']['he_rgb'][:2]))
expected_hsi_size = tuple(reversed(metadata['shapes']['hsi_rgb'][:2]))
assert he_image.size == expected_he_size
assert hsi_image.size == expected_hsi_size


def load_font(size, bold=False):
    candidates = [
        Path(r'C:\Windows\Fonts\arialbd.ttf' if bold else r'C:\Windows\Fonts\arial.ttf'),
        Path(r'C:\Windows\Fonts\calibrib.ttf' if bold else r'C:\Windows\Fonts\calibri.ttf'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return ImageFont.truetype(str(candidate), size=size)
    return ImageFont.load_default()


def add_scale_bar(image, bar_length_px, label='1 cm'):
    output = image.copy()
    draw = ImageDraw.Draw(output)
    x1 = 24
    y = output.height - 28
    x2 = x1 + int(round(bar_length_px))
    tick = 9

    # El trazo negro mejora la lectura sobre zonas claras y oscuras del tejido.
    draw.line((x1, y, x2, y), fill='black', width=8)
    draw.line((x1, y - tick, x1, y + tick), fill='black', width=8)
    draw.line((x2, y - tick, x2, y + tick), fill='black', width=8)
    draw.line((x1, y, x2, y), fill=(255, 235, 0), width=4)
    draw.line((x1, y - tick, x1, y + tick), fill=(255, 235, 0), width=4)
    draw.line((x2, y - tick, x2, y + tick), fill=(255, 235, 0), width=4)

    label_font = load_font(20, bold=True)
    draw.text(
        (x1, y - 34),
        label,
        font=label_font,
        fill=(255, 235, 0),
        stroke_width=3,
        stroke_fill='black',
    )
    return output


he_with_bar = add_scale_bar(he_image, px_per_cm)
hsi_with_bar = add_scale_bar(hsi_image, px_per_cm)

margin_x = 36
gap = 88
title_height = 64
label_height = 42
bottom_margin = 32
content_height = max(he_with_bar.height, hsi_with_bar.height)
canvas_width = margin_x * 2 + he_with_bar.width + gap + hsi_with_bar.width
canvas_height = title_height + label_height + content_height + bottom_margin
canvas = Image.new('RGB', (canvas_width, canvas_height), 'white')
draw = ImageDraw.Draw(canvas)

title_font = load_font(24, bold=False)
panel_font = load_font(22, bold=True)
scale_label = f'{px_per_cm:.1f}'.replace('.', ',')
title = f'SB013: imágenes segmentadas a escala física común ({scale_label} px/cm)'
title_bbox = draw.textbbox((0, 0), title, font=title_font)
draw.text(
    ((canvas_width - (title_bbox[2] - title_bbox[0])) / 2, 18),
    title,
    font=title_font,
    fill=(20, 20, 20),
)

he_x = margin_x
hsi_x = margin_x + he_with_bar.width + gap
image_top = title_height + label_height
he_y = image_top + (content_height - he_with_bar.height) // 2
hsi_y = image_top + (content_height - hsi_with_bar.height) // 2

draw.text((he_x, title_height + 7), f'H&E segmentada: {he_image.width} × {he_image.height} px', font=panel_font, fill=(20, 20, 20))
draw.text((hsi_x, title_height + 7), f'HSI segmentada: {hsi_image.width} × {hsi_image.height} px', font=panel_font, fill=(20, 20, 20))
canvas.paste(he_with_bar, (he_x, he_y))
canvas.paste(hsi_with_bar, (hsi_x, hsi_y))

# Se amplía el lienzo completo, no cada modalidad por separado, para conservar la escala relativa.
output_scale = 2
canvas = canvas.resize(
    (canvas.width * output_scale, canvas.height * output_scale),
    Image.Resampling.LANCZOS,
)
figure_path.parent.mkdir(parents=True, exist_ok=True)
canvas.save(figure_path, dpi=(300, 300))

print('Figura guardada en:', figure_path)
print('H&E final (ancho × alto):', he_image.size, 'px')
print('HSI final (ancho × alto):', hsi_image.size, 'px')
print('Escala común:', round(px_per_cm, 4), 'px/cm')
print('Barra de 1 cm:', round(px_per_cm), 'px antes de ampliar el lienzo')
